# ST-CDGM **V8** — run GPU complet

Mêmes jeux de données que depuis le début. Aucun prétraitement à lancer : les
**13 nœuds libres** sont dérivés à la volée depuis les 15 canaux bruts.

## Ce qui change par rapport à tous les runs précédents

| | avant | V8 |
|---|---|---|
| nœuds du DAG | plongements de métachemins, **mêmes entrées pour tous** | 13 quantités physiques, **un canal chacun** |
| drivers | un embedding partagé + un biais appris | **routage diagonal** : l'info inter-variable ne passe que par `A[u,v]` |
| structure | `A_dag` mono-lag | `A_inst` (τ=0) **et** `A_dag` (τ≥1) |
| prior | MSE vers la matrice complète | **C7** : 3 niveaux, λ annelé par niveau, hors-prior LIBRE |
| décodeur | requêtes apprises, aveugles à l'entrée | **A2a** : requêtes amorcées par l'état + encodage positionnel partagé |
| perte étage 1 | MSE sur log1p | **A3** : vraisemblance Bernoulli-Gamma, ancre `μ = p·α·β` |
| retour en mm | `expm1(μ)` | **A1** : `expm1(μ + s²/2)`, `s²` hétéroscédastique |

## Pourquoi ces changements

Trois audits ont trouvé le même défaut à trois endroits : **les variables ne se
distinguaient jamais par leurs entrées**, seulement par des poids appris. Les
types de nœuds du builder recevaient tous les mêmes 15 canaux ; `driver_encoder`
distribuait le même vecteur aux q variables ; et aucune ligne du dépôt ne
calculait l'IVT. Un DAG dans ces conditions est décoratif : rien ne le rend
load-bearing. V8 corrige les trois.

## Interrupteurs

Chaque brique s'éteint séparément (`V8` en Cell 2). P1 demande **un seul
changement par run** pour pouvoir attribuer l'effet. Tout à `False` ≈ pile V5.

## Cible

**Parité in-distribution + gain OOD.** Une régression ID de quelques pour cent
est prévue et acceptée : le DAG gelé est une feature OOD assumée, pas un
avantage ID. Le verdict se joue sur EC-Earth3, puis **une seule fois** sur le
holdout.

## Comparaison aux autres modèles

Les Cells 10-11 mesurent V8 **dans le protocole du 3-way** (K=32, 24 pas,
cfg 0.0, split de test complet, composition mm par membre, `evaluate_ensemble`)
puis relisent la table déjà produite pour V6', ORACLE (V5) et CorrDiff : ces
trois-là ne sont **pas réévalués**. C'est aussi pourquoi l'étage 2 est l'UNet
CorrDiff-Normal (~43 M paramètres) et non l'UNet minimal du YAML de base : à
1 M paramètres, la table comparerait des tailles de réseau.

Si la table de référence est absente, la Cell 11 le dit et marque la
comparaison non valide plutôt que de s'appuyer sur les valeurs
pré-enregistrées, que le prérégistre V6' déclare lui-même calculées avec une
Convention A buggée.

## Règle absolue

**NorESM2-MM est le holdout OOD.** Aucune cellule ne l'ouvre. Un garde lève à
la moindre tentative.

In [ ]:
# >>> Cell 1 : bootstrap Colab + montage Drive
import os, sys, json, math, time, warnings, subprocess
from pathlib import Path
warnings.filterwarnings("ignore")

GIT_URL    = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH = "four-node-causal"
REPO_DIR   = "/content/climate_data"
DRIVE_ROOT = "/content/drive/MyDrive/climate_data"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Les .nc d'entrainement (3 Go) sont gitignores : ils vivent sur Drive,
    # pas dans le depot. Sans ce montage, la Cell 2 s'arrete faute de donnees.
    if not Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")

    if not Path(REPO_DIR).exists():
        # Clone sur le SSD local, jamais sur Drive : xarray y est ~20x plus lent.
        subprocess.check_call(["git", "clone", "--depth=200", "-b", GIT_BRANCH,
                               GIT_URL, REPO_DIR])
    else:
        subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin"])
        subprocess.check_call(["git", "-C", REPO_DIR, "checkout", GIT_BRANCH])
        subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", GIT_BRANCH])
    # Liste EPINGLEE, reprise telle quelle du 9-node et de V6' qui tournaient.
    # Une liste courte et non epinglee produit la cascade "une erreur par run" :
    #   cftime            -> decodage du calendrier 'noleap' de nos predicteurs
    #                        (sinon crash sur TOUT open NetCDF)
    #   netcdf4/h5netcdf  -> moteurs NetCDF-4. Sans eux xarray se rabat sur
    #                        scipy, qui ne lit que le NetCDF-3, et rejette nos
    #                        fichiers avec "is not a valid NetCDF 3 file" —
    #                        message trompeur : le fichier est bon.
    #   xbatcher          -> NetCDFDataPipeline.__init__ leve ImportError sans lui
    #   diffusers==0.36.0 + la pile epinglee -> UNet2DConditionModel stable
    # Le try/except evite de reinstaller a chaque relance de la cellule.
    try:
        import torch_geometric, cftime, h5netcdf, xbatcher, diffusers, omegaconf  # noqa: F401,E401
        print("deps critiques presentes — pip install saute.")
    except Exception as _e:
        print(f"pip install requis : {_e}")
        _DEPS = ["omegaconf==2.3.0", "hydra-core==1.3.2", "diffusers==0.36.0",
                 "transformers==4.57.6", "accelerate==1.12.0",
                 "huggingface-hub==0.36.0", "safetensors==0.7.0",
                 "xbatcher", "webdataset", "cftime", "h5netcdf", "netcdf4",
                 "numcodecs", "scipy", "torch-geometric", "xformers"]
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--no-warn-script-location", *_DEPS])
    ROOT = Path(REPO_DIR)

    # Verification que le code V8 est REELLEMENT arrive. Si la branche n'a pas
    # ete poussee, le clone rend une version anterieure et le notebook echoue
    # 4 cellules plus loin sur un ImportError incomprehensible.
    _requis = ["src/st_cdgm/data/derived.py", "src/st_cdgm/priors.py",
               "src/st_cdgm/models/bernoulli_gamma.py",
               "src/st_cdgm/evaluation/jensen.py", "config/dag_prior_v8_c7.yaml"]
    _absents = [f for f in _requis if not (ROOT / f).exists()]
    if _absents:
        _head = subprocess.check_output(
            ["git", "-C", REPO_DIR, "log", "-1", "--oneline"]).decode().strip()
        raise RuntimeError(
            "Le code V8 n'est pas dans la branche clonee. Manquants : "
            + ", ".join(_absents)
            + f" | HEAD = {_head} | branche = {GIT_BRANCH}. "
            "Pousser la branche (git push origin " + GIT_BRANCH + ") "
            "avant de relancer.")
else:
    ROOT = Path.cwd()

# chdir a la racine : les chemins relatifs (config/*.yaml, data/, checkpoints/)
# echouent sinon depuis /content.
os.chdir(ROOT)
for _p in (str(ROOT), str(ROOT / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Verifier qu'un moteur NetCDF-4 est reellement disponible AVANT la Cell 3 :
# sinon l'erreur ne surgit qu'a l'ouverture du fichier, avec un message qui
# accuse le fichier au lieu de l'environnement.
import importlib
_moteurs = [m for m in ("netCDF4", "h5netcdf") if importlib.util.find_spec(m)]
_cal = importlib.util.find_spec("cftime") is not None
if not _moteurs or not _cal:
    raise ImportError(
        f"Environnement incomplet — moteurs NetCDF-4 : {_moteurs or 'AUCUN'}, "
        f"cftime : {'oui' if _cal else 'NON'}. Sans moteur NetCDF-4 xarray se "
        "rabat sur scipy (NetCDF-3 seulement) et accuse le fichier ; sans "
        "cftime le calendrier 'noleap' de nos predicteurs ne se decode pas. "
        "Installer : pip install netcdf4 h5netcdf cftime, puis relancer.")
print(f"moteurs NetCDF : {_moteurs} | cftime : oui")

import numpy as np, torch
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"racine  : {ROOT}")
print(f"torch {torch.__version__} | device = {DEVICE}")
if DEVICE.type == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print(f"  {_p.name} | {_p.total_memory / 2**30:.1f} GiB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("  ATTENTION : prevu pour GPU. L'etage 2 sera tres lent sur CPU.")

In [ ]:
# >>> Cell 2 : configuration V8 + interrupteurs + garde holdout
from omegaconf import OmegaConf
from st_cdgm.data.derived import FREE_NODES

CONFIG = OmegaConf.load("config/training_config.yaml")

# --- Etage 2 : la MEME architecture que les modeles auxquels on se compare.
# Les metriques de reference (ORACLE, CorrDiff, V6') ont toutes ete produites
# avec l'UNet CorrDiff-Normal : 4 niveaux, [128,256,256,256], ~50 M parametres
# (champ `config_block_out_channels` de leurs JSON de metriques). Le bloc
# `diffusion` du YAML de base decrit un UNet MINIMAL de 1 M parametres : le
# garder ferait mesurer la taille de l'UNet, pas l'apport de V8. Le merge
# apporte aussi S_churn=40 et la tail_weight 8/25 du protocole de reference.
_S2_REF = OmegaConf.load("config/training_config_corrdiff_normal.yaml")
CONFIG.diffusion = OmegaConf.merge(CONFIG.diffusion, _S2_REF.diffusion)
print(f"etage 2 : UNet {list(CONFIG.diffusion.unet_kwargs.block_out_channels)} "
      f"(CorrDiff-Normal, celui des references)")

# --- Interrupteurs. Tout a False donne approximativement la pile V5.
#     P1 exige UN SEUL changement par run pour pouvoir attribuer l'effet.
V8 = OmegaConf.create(dict(
    free_nodes      = True,   # 13 noeuds libres derives au lieu des 15 bruts
    diagonal_driver = True,   # V7-M2 : chaque variable ne voit que son canal
    instantaneous   = True,   # C2/C5 : A(0) contemporaine en plus de A(tau>=1)
    edge_prior      = True,   # C7 : prior 3 niveaux, annele par niveau
    spatial_queries = True,   # A2a : requetes du decodeur dependantes de l'entree
    bernoulli_gamma = True,   # A3 : vraisemblance BG, ancre mu = p*alpha*beta
    jensen          = True,   # A1 : correction du retour log1p -> mm
))
print(OmegaConf.to_yaml(V8))

# --- Budget. Pour un smoke, reduire EPOCHS ; ne JAMAIS toucher aux seuils.
EPOCHS_S1   = int(os.environ.get("V8_EPOCHS_S1", 30))
# Etage 2 : le budget se compte en TIRAGES (echantillons vus), la seule unite
# invariante au batch ET au stride. V5 : 250 epoques x 5 467 fenetres = 1,37 M.
# EPOCHS_S2 en decoule a la Cell 9, une fois le cache connu — le fixer ici
# serait refaire l'erreur d'origine, ou 20 epoques (le cap `stage2.epochs_max`
# du YAML de base) ne donnaient que ~55 000 tirages, vingt-cinq fois moins.
TARGET_DRAWS_S2 = int(os.environ.get("V8_DRAWS_S2", 1_370_000))
N_EVAL      = int(os.environ.get("V8_N_EVAL", 300))   # audit Jensen (Cell 7)

# --- Protocole d'evaluation. Ces valeurs ne sont PAS libres : ce sont celles
#     sous lesquelles V6', ORACLE (V5) et CorrDiff ont ete mesures dans le
#     3-way. Les changer rend la Cell 11 incomparable — elle le detecte et
#     refuse le verdict plutot que d'aligner des nombres de protocoles
#     differents. cfg 0.0 = conditioned-only : identique a 1.0 sur edm_karras,
#     sans le double forward CFG.
K_VERDICT   = int(os.environ.get("V8_K", 32))
NUM_STEPS   = int(os.environ.get("V8_NUM_STEPS", 24))
CFG_SCALE   = 0.0
EVAL_BATCH  = int(os.environ.get("V8_EVAL_BATCH", 16))

# --- Stride. Les trois references ont tourne a stride 2 (`data.stride` de
#     training_config_corrdiff_normal.yaml, repris tel quel par V5 et V6').
#     L'EVALUATION doit s'y aligner : a stride 4 le split de test ne contient
#     que la moitie des fenetres, et le seuil poole de la Convention B ne
#     porterait pas sur le meme echantillon.
#     L'ENTRAINEMENT reste a 4 : le budget en tirages est deja apparie, seule
#     la diversite du cache differe (~2 734 fenetres distinctes contre 5 467).
#     Passer a 2 double le cout de l'etage 1 et laisse celui de l'etage 2
#     inchange (les epoques se divisent par deux, a tirages constants).
STRIDE_TRAIN = int(os.environ.get("V8_STRIDE_TRAIN", CONFIG.data.get("stride", 4)))
STRIDE_EVAL  = int(os.environ.get("V8_STRIDE_EVAL", _S2_REF.data.stride))
print(f"stride : entrainement={STRIDE_TRAIN} | evaluation={STRIDE_EVAL} "
      f"(references : {int(_S2_REF.data.stride)})")
SEQ_LEN     = int(CONFIG.data.seq_len)
HIDDEN      = int(CONFIG.rcn.hidden_dim)
LR_SHAPE    = (23, 26)
HR_SHAPE    = (172, 179)
# Checkpoints sur DRIVE en Colab. `checkpoints_v8/` sous /content est efface
# avec la VM : sur un run de plusieurs heures, une deconnexion perdrait
# tout. Le clone du code reste sur le SSD local (xarray y est ~20x plus
# rapide), mais les poids doivent survivre a la session.
CKPT_DIR = ((Path(DRIVE_ROOT) / "checkpoints_v8") if IN_COLAB
            else Path("checkpoints_v8"))
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f"checkpoints : {CKPT_DIR}")
# Les resultats aussi sur Drive, et pour la meme raison que les checkpoints :
# `results/` vit dans le clone, donc sur le disque EPHEMERE de la VM. Tout y
# passait — l'historique des pertes, la climatologie, les metriques
# in-protocol, le verdict. Le cache de metriques de la Cell 10, cense eviter
# de reechantillonner pendant des dizaines de minutes, ne survivait meme pas a
# une deconnexion.
RESULTS_DIR = ((Path(DRIVE_ROOT) / "results_v8") if IN_COLAB
               else Path("results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"resultats   : {RESULTS_DIR}")

# --- Garde holdout. NorESM2-MM est pre-enregistre comme intouchable : une
#     seule evaluation finale, jamais pendant le developpement. Toute cellule
#     qui ouvre un fichier passe par guard().
HOLDOUT = "NorESM2"


def guard(p):
    assert HOLDOUT not in str(p), f"STOP - {p} touche le holdout {HOLDOUT}-MM."
    return Path(p)


print("13 noeuds :", list(FREE_NODES))
print("Les chemins de donnees sont resolus en Cell 2b (telechargement si absent).")

In [ ]:
# >>> Cell 2b : approvisionnement des donnees - cherche puis TELECHARGE
# Les .nc d'entrainement (3 Go) sont gitignores : ils ne viennent pas du clone.
# Cette cellule rend le notebook autonome, comme la Cell 2B des runs precedents.
import glob
import urllib.request as _ureq
import urllib.error as _uerr

DATA_ROOT = Path(f"{DRIVE_ROOT}/data") if IN_COLAB else Path("data/raw")
for _sub in ("train", "static_predictors"):
    (DATA_ROOT / _sub).mkdir(parents=True, exist_ok=True)

# Depot Zenodo du jeu NIWA/CCAM utilise depuis le debut. ACCESS-CM2 seulement :
# les autres GCM (OOD) ne sont pas sur Zenodo et doivent etre deposes a la main.
ZENODO = {
    "predictor_ACCESS-CM2_hist.nc":
        "https://zenodo.org/records/10889046/files/predictor_ACCESS-CM2_hist.nc?download=1",
    "pr_ACCESS-CM2_hist.nc":
        "https://zenodo.org/records/10889046/files/pr_ACCESS-CM2_hist.nc?download=1",
}


def stream_dl(url, dest, retries=5, chunk=1024 * 1024):
    """Telechargement avec REPRISE (header Range) et backoff exponentiel.

    3 Go sur une session Colab : une coupure reseau est probable, pas
    exceptionnelle. Sans reprise il faudrait tout recommencer. Le fichier est
    ecrit en .part puis renomme : un fichier tronque ne peut pas etre pris pour
    un telechargement reussi au run suivant.
    """
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, retries + 1):
        already = part.stat().st_size if part.exists() else 0
        req = _ureq.Request(url)
        if already > 0:
            req.add_header("Range", f"bytes={already}-")
            print(f"    reprise a {already / 1e6:.1f} MB")
        try:
            with _ureq.urlopen(req, timeout=30) as resp:
                # Longueur ANNONCEE pour CETTE requete (206 -> ce qui reste).
                _cl = resp.headers.get("Content-Length")
                attendu = int(_cl) if _cl is not None else None
                with open(part, "ab" if already > 0 else "wb") as f:
                    got, last, last_b = already, time.time(), already
                    while True:
                        c = resp.read(chunk)
                        if not c:
                            break
                        f.write(c)
                        got += len(c)
                        if time.time() - last >= 5:
                            sp = (got - last_b) / (time.time() - last) / 1e6
                            print(f"    {got / 1e6:8.1f} MB  --  {sp:5.1f} MB/s")
                            last, last_b = time.time(), got
            # Un serveur qui ferme la connexion en cours de route ne leve PAS :
            # read() rend vide, la boucle sort normalement, et sans ce controle
            # le .part tronque serait promu comme un fichier complet. C'est le
            # mode de defaillance le plus couteux ici : open_dataset accepterait
            # un .nc mutile et rendrait des NaN plusieurs cellules plus loin.
            if attendu is not None and (got - already) != attendu:
                raise ConnectionError(
                    f"flux tronque : {got - already} octets recus sur "
                    f"{attendu} annonces (le .part est conserve pour la reprise)")
            os.replace(part, dest)
            print(f"  OK {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)")
            return dest
        except (_uerr.HTTPError, _uerr.URLError, TimeoutError, ConnectionError) as e:
            wait = min(60, 2 ** attempt)
            print(f"  WARN {type(e).__name__}: {e} -- nouvel essai dans {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"echec du telechargement apres {retries} essais : {url}")


def find(patterns, roots):
    """Recherche recursive : le fichier peut etre range differemment sur Drive."""
    for r in roots:
        for pat in patterns:
            hits = sorted(glob.glob(str(Path(r) / "**" / pat), recursive=True))
            if hits:
                return Path(hits[0])
    return None


ROOTS = [Path("data/raw"), DATA_ROOT] + ([Path(DRIVE_ROOT)] if IN_COLAB else [])

LR_PATH = find(["predictor_ACCESS-CM2_hist.nc"], ROOTS)
HR_PATH = find(["pr_ACCESS-CM2_hist.nc"], ROOTS)
# Le statique EST suivi par git : il arrive avec le clone.
STATIC_PATH = find(["*NZ_Invariant.nc", "ERA5_eval_ccam_12km*.nc"], ROOTS)

if LR_PATH is None:
    print("[2b] predicteurs absents -> Zenodo")
    LR_PATH = stream_dl(ZENODO["predictor_ACCESS-CM2_hist.nc"],
                        DATA_ROOT / "train" / "predictor_ACCESS-CM2_hist.nc")
if HR_PATH is None:
    print("[2b] precipitation HR absente -> Zenodo (2,5 Go, comptez ~10 min)")
    HR_PATH = stream_dl(ZENODO["pr_ACCESS-CM2_hist.nc"],
                        DATA_ROOT / "train" / "pr_ACCESS-CM2_hist.nc")
if STATIC_PATH is None:
    raise FileNotFoundError(
        "Statique HR (orographie, masque terre/mer) introuvable. Il est suivi "
        "par git et devrait arriver avec le clone : verifier "
        "data/raw/static_predictors/ dans le depot.")

for _p in (LR_PATH, HR_PATH, STATIC_PATH):
    guard(_p)

# Verification de taille : un .nc tronque passerait open_dataset et donnerait
# des NaN silencieux plusieurs cellules plus loin.
for _n, _p, _min_mb in (("LR", LR_PATH, 500), ("HR", HR_PATH, 2000)):
    _mb = _p.stat().st_size / 1e6
    if _mb < _min_mb:
        raise RuntimeError(
            f"{_n} = {_p} ne fait que {_mb:.0f} MB (attendu > {_min_mb} MB) : "
            f"telechargement incomplet, supprimer le fichier et relancer.")

print(f"LR      : {LR_PATH}  ({LR_PATH.stat().st_size / 1e6:.0f} MB)")
print(f"HR      : {HR_PATH}  ({HR_PATH.stat().st_size / 1e6:.0f} MB)")
print(f"statique: {STATIC_PATH}")

In [ ]:
# >>> Cell 3 : pipeline - les 13 noeuds derives a la volee
from st_cdgm.data.pipeline import NetCDFDataPipeline

# lr_free_nodes=True : la derivation est faite UNE fois sur le jeu complet, pas
# par fenetre. Tout l'aval (_dataset_to_numpy, lr_grid_to_nodes, le driver du
# RCN) voit alors 13 canaux dans l'ordre de FREE_NODES, ce qui est exactement
# la carte identite du routage diagonal.
t0 = time.time()
pipeline = NetCDFDataPipeline(
    lr_path=guard(LR_PATH), hr_path=guard(HR_PATH), static_path=guard(STATIC_PATH),
    seq_len=SEQ_LEN,
    baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.precipitation_delta),
    lr_free_nodes=bool(V8.free_nodes),
    lr_variables=None if V8.free_nodes else list(CONFIG.data.lr_variables),
    hr_variables=list(CONFIG.data.hr_variables),
    static_variables=(list(CONFIG.data.static_variables)
                      if CONFIG.data.get("static_variables") else []),
    train_start_date=CONFIG.data.get("train_start_date"),
    train_end_date=CONFIG.data.get("train_end_date"),
    val_start_date=CONFIG.data.get("val_start_date"),
    val_end_date=CONFIG.data.get("val_end_date"),
    test_start_date=CONFIG.data.get("test_start_date"),
    test_end_date=CONFIG.data.get("test_end_date"),
)
LR_VARS = list(pipeline.get_lr_dataset().data_vars)
print(f"[{time.time() - t0:.0f}s] canaux LR ({len(LR_VARS)}) : {LR_VARS}")

if V8.free_nodes:
    # L'ordre est load-bearing : une permutation silencieuse ferait tourner le
    # modele en associant chaque variable au mauvais champ, sans rien signaler.
    assert LR_VARS == list(FREE_NODES), (
        f"ordre des canaux != FREE_NODES\n  recu   : {LR_VARS}\n  attendu: {list(FREE_NODES)}")

# stride : sans lui build_sequence_dataset retombe sur 1, soit 10 935
# echantillons par epoque au lieu des ~2 734 configures. Quatre fois le temps
# de calcul ET quatre fois le cache de l'etage 2 - c'est ce qui fait passer
# l'empreinte memoire de 1,1 Go a 4,4 Go, donc d'un run qui tient sur T4 a un
# run qui sature.
# Le split de TEST prend le stride des references (Cell 2) : les metriques
# auxquelles on se compare ont ete calculees sur ces fenetres-la.
train_dataset = pipeline.build_sequence_dataset(split="train", stride=STRIDE_TRAIN,
                                                training=True)
val_dataset   = pipeline.build_sequence_dataset(split="val", stride=STRIDE_TRAIN)
test_dataset  = pipeline.build_sequence_dataset(split="test", stride=STRIDE_EVAL)
print(f"stride : train/val={STRIDE_TRAIN} | test={STRIDE_EVAL}")

_s = next(iter(train_dataset))
print("echantillon : lr", tuple(_s["lr"].shape),
      "| residual", tuple(_s["residual"].shape),
      "| baseline", tuple(_s["baseline"].shape))
assert _s["lr"].shape[1] == len(LR_VARS)

In [ ]:
# >>> Cell 4 : la pile V8
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder
from st_cdgm.models.bernoulli_gamma import BernoulliGammaHead
from st_cdgm.priors import load_edge_prior, DEFAULT_LEVEL_FLOORS

torch.manual_seed(SEED)

# --- graphe : 13 types de noeuds REELLEMENT distincts ----------------------
# free_nodes_v8 remplace entierement le jeu de noeuds (pas de GP850/500/250).
builder = HeteroGraphBuilder(
    lr_shape=LR_SHAPE, hr_shape=HR_SHAPE,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=not bool(V8.free_nodes),
    free_nodes_v8=bool(V8.free_nodes),
)
hetero_template, _report = builder.build()
NODE_TYPES = list(builder.dynamic_node_types)
Q = len(NODE_TYPES)
print(f"q = {Q} variables : {NODE_TYPES}")

# --- encodeur : un metachemin spatial par variable -------------------------
enc_cfgs = [IntelligibleVariableConfig(name=f"{n}_spat",
                                       meta_path=(n, "spat_adj", n), pool="mean")
            for n in NODE_TYPES]
encoder = IntelligibleVariableEncoder(
    configs=enc_cfgs, hidden_dim=HIDDEN,
    conditioning_dim=int(CONFIG.encoder.conditioning_dim)).to(DEVICE)

# MATERIALISATION OBLIGATOIRE. L'encodeur est bati sur des LazyModule : ses
# poids n'existent qu'apres un premier forward, et il faut un graphe PORTANT
# DES FEATURES (le template n'en a pas). Les notebooks precedents ne s'en
# apercevaient pas parce qu'ils chargeaient un checkpoint - load_state_dict
# materialise. V8 part de zero : sans ce forward a blanc, encoder.parameters()
# serait VIDE au moment de construire l'optimiseur, et l'encodeur resterait a
# son initialisation aleatoire pendant tout l'entrainement, EN SILENCE.
from st_cdgm.evaluation.evaluation_xai import convert_sample_to_batch

def convert_sample_v8(sample, bld=None, dev=None):
    """Comme convert_sample_to_batch, mais chaque noeud recoit SON canal.

    Le convertisseur partage fait :
        dynamic_features = {nt: lr_nodes[0] for nt in dynamic_node_types}
    c'est-a-dire le MEME tenseur 13 canaux pour les 13 types. L'etat initial
    H(0) ne distingue donc les variables que par les poids de convolution de
    leur metachemin - exactement le defaut que V8 corrige ailleurs. Le routage
    diagonal V7-M2 ne couvre que la RECURRENCE ; sans ce correctif, le point de
    depart de la recurrence reste indifferencie.

    Ici le noeud d'indice v recoit uniquement le canal v, dans l'ordre de
    FREE_NODES - la meme carte identite que le routage diagonal.
    """
    bld = bld if bld is not None else builder
    dev = dev if dev is not None else DEVICE
    lr_seq = sample["lr"]
    steps = [bld.lr_grid_to_nodes(lr_seq[t]) for t in range(lr_seq.shape[0])]
    if V8.free_nodes:
        first = steps[0]                                   # [N_lr, 13]
        feats = {nt: first[:, i:i + 1] for i, nt in enumerate(NODE_TYPES)}
    else:
        feats = {nt: steps[0] for nt in bld.dynamic_node_types}
    return {"lr": torch.stack(steps, dim=0),
            "residual": sample["residual"], "baseline": sample.get("baseline"),
            "hetero": bld.prepare_step_data(feats).to(dev)}

with torch.no_grad():
    _warm = convert_sample_v8(_s, builder, DEVICE)
    _H0 = encoder.init_state(_warm["hetero"])
_n_enc = sum(p.numel() for p in encoder.parameters())
assert _n_enc > 0, "encodeur non materialise : l'optimiseur serait vide"
assert _H0.shape[0] == Q, f"H_init a {_H0.shape[0]} variables, attendu {Q}"
print(f"encodeur  : {_n_enc:,} parametres materialises | H_init {tuple(_H0.shape)}")

# --- prior C7 --------------------------------------------------------------
edge_prior, A_prior, A_inst_prior = None, None, None
if V8.edge_prior and V8.free_nodes:
    edge_prior = load_edge_prior()
    assert len(edge_prior.nodes) == Q, (
        f"prior sur {len(edge_prior.nodes)} noeuds, graphe a {Q} variables")
    # node_order : l'ordre du RCN n'a aucune raison d'etre celui du YAML.
    # FILTRAGE PAR LAG obligatoire : A_dag est A(1), A_inst est A(0). Initialiser
    # A(1) avec TOUTES les aretes y injecterait les 24 aretes contemporaines que
    # la perte route pourtant vers A(0) — et A(0) demarrerait au bruit, sans
    # prior, alors que c'est la structure que C2/C5 existe pour exprimer.
    A_prior = torch.as_tensor(edge_prior.matrix(lag=1, node_order=NODE_TYPES))
    A_inst_prior = (torch.as_tensor(edge_prior.matrix(lag=0, node_order=NODE_TYPES))
                    if V8.instantaneous else None)
    print(f"prior C7  : {len(edge_prior)} aretes {edge_prior.by_level()}")
    print(f"            orientation {edge_prior.by_orient()}")
    print(f"            init : A(1) <- {len(edge_prior.edge_list(lag=1))} aretes | "
          f"A(0) <- {len(edge_prior.edge_list(lag=0))} aretes")
    if not V8.instantaneous and edge_prior.edge_list(lag=0):
        raise ValueError(
            f"{len(edge_prior.edge_list(lag=0))} aretes du prior sont a lag 0 "
            f"mais V8.instantaneous=False : elles seraient ecrasees sur A(1).")

# --- RCN : routage diagonal (V7-M2) + A(0) contemporaine -------------------
rcn_cell = RCNCell(
    num_vars=Q, hidden_dim=HIDDEN, driver_dim=len(LR_VARS),
    reconstruction_dim=len(LR_VARS),
    dropout=float(CONFIG.rcn.dropout),
    dag_prior=A_prior, inst_prior=A_inst_prior,
    instantaneous=bool(V8.instantaneous),
    driver_routing=("diagonal" if (V8.diagonal_driver and V8.free_nodes) else "shared"),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))
print(f"RCN       : routage={rcn_cell.driver_routing} | "
      f"A(0)={'oui' if rcn_cell.A_inst is not None else 'non'}")

# --- decodeur : requetes spatiales (A2a) -----------------------------------
_rh = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=HIDDEN, hr_h=HR_SHAPE[0], hr_w=HR_SHAPE[1],
    intermediate_h=int(_rh.intermediate_h), intermediate_w=int(_rh.intermediate_w),
    n_heads=int(_rh.n_heads), refine_channels=int(_rh.refine_channels),
    query_mode=("spatial" if V8.spatial_queries else "learned"),
    lr_h=LR_SHAPE[0], lr_w=LR_SHAPE[1],
).to(DEVICE)
print(f"decodeur  : query_mode={regression_head.query_mode} | "
      f"features={regression_head.feature_channels}")

# --- tete Bernoulli-Gamma (A3) --------------------------------------------
bg_head = (BernoulliGammaHead(regression_head.feature_channels).to(DEVICE)
           if V8.bernoulli_gamma else None)
# A3 exclut le melange convexe du skip-block : A4 propose de le retirer, et il
# casserait l'interpretation de mu = p*alpha*beta comme moyenne conditionnelle.
skip_block = None

STAGE1_MODULES = [encoder, rcn_cell, regression_head] + ([bg_head] if bg_head else [])
n_par = sum(p.numel() for m in STAGE1_MODULES for p in m.parameters())
print(f"parametres etage 1 : {n_par:,}")

In [ ]:
# >>> Cell 5 : entrainement etage 1
from torch.optim import AdamW
from st_cdgm.training.training_loop import train_epoch_stage1

def iterate_batches(ds, bld=None, dev=None):
    """Signature (data_loader, builder, device) attendue par les helpers."""
    for s in ds:
        yield [convert_sample_v8(s, bld, dev)]

# bg_head fait partie des parametres optimises : l'oublier laisserait la tete
# a son initialisation et la NLL ne descendrait jamais. L'encodeur, lui, a ete
# materialise en Cell 4 - sinon ses poids lazy seraient absents d'ici.
assert all(sum(1 for _ in m.parameters()) > 0 for m in STAGE1_MODULES),     "un module n'expose aucun parametre - voir la materialisation en Cell 4"
# Les hyperparametres de l'etage 1 vivent sous two_stage.stage1, PAS a la
# racine de `training`. Les lire avec .get() depuis le mauvais niveau
# retomberait en silence sur des defauts codes en dur — le YAML serait ignore
# sans que rien ne le signale.
S1 = CONFIG.two_stage.stage1
opt_s1 = AdamW([p for m in STAGE1_MODULES for p in m.parameters()],
               lr=float(S1.lr), betas=(0.9, 0.99),
               weight_decay=float(S1.get("weight_decay", 0.0)))
print(f"etage 1 : lr={float(S1.lr):.1e} lambda_reg={float(S1.lambda_reg)} "
      f"beta_rec={float(S1.beta_rec)} gamma_dag_max={float(S1.gamma_dag_max)} "
      f"lambda_l1={float(S1.lambda_l1)}")

# Une perte de validation. Sans elle, 30 epoques tournent a l'aveugle et le
# "meilleur" checkpoint choisi sur la perte d'ENTRAINEMENT revient a prendre la
# derniere epoque — aucune protection contre le sur-apprentissage, alors que le
# collapse du DAG et l'apprentissage du relief au lieu de la meteo sont des
# modes de defaillance documentes de cette architecture.
from st_cdgm.models.bernoulli_gamma import decode_bg_params, stage1_bg_loss

# UN seul seuil humide, lu aux deux endroits. Le laisser en dur a deux endroits
# est exactement ce qui a fait diverger l'entrainement (0,1) de la validation
# (defaut 1,0) sans que rien ne le signale.
BG_WET_SEUIL = 0.1


@torch.no_grad()
def validation_loss(ds):
    for _m in STAGE1_MODULES:
        _m.eval()
    tot, n = 0.0, 0
    for s_ in ds:
        b = convert_sample_v8(s_, builder, DEVICE)
        t = b["residual"][-1].to(DEVICE)
        if t.dim() == 3:
            t = t.unsqueeze(0)
        bl = b["baseline"][-1].to(DEVICE)
        if bl.dim() == 3:
            bl = bl.unsqueeze(0)
        lr_data = b["lr"].to(DEVICE)
        H0 = encoder.init_state(b["hetero"])
        seq = rcn_runner.run(H0, [lr_data[k] for k in range(lr_data.shape[0])],
                             reconstruction_sources=None)
        H_T = seq.states[-1]
        if bg_head is not None:
            pb, ab, bb = decode_bg_params(regression_head, bg_head, H_T,
                                          target_shape=t.shape[-2:])
            # wet_threshold EXPLICITE. Le defaut de stage1_bg_loss est 1,0 et
            # l'entrainement passe 0,1 : sans cet argument, la validation
            # notait une AUTRE vraisemblance que celle optimisee — elles
            # divergent sur toute la bande 0,1 a 1 mm/j, precisement celle que
            # A3 existe pour traiter. Le checkpoint "meilleur" etait donc
            # selectionne sur un critere que le modele n'a jamais minimise, et
            # la courbe de validation plate n'etait pas interpretable.
            v, _ = stage1_bg_loss(pb, ab, bb, t, bl, wet_threshold=BG_WET_SEUIL)
        else:
            mu = regression_head(H_T)
            if mu.shape != t.shape:
                mu = torch.nn.functional.interpolate(
                    mu, size=t.shape[-2:], mode="bilinear", align_corners=False)
            m_ = torch.isfinite(t)
            v = ((mu - torch.nan_to_num(t))[m_] ** 2).mean()
        tot += float(v)
        n += 1
    for _m in STAGE1_MODULES:
        _m.train()
    return tot / max(n, 1)


# REPRISE. Un run de plusieurs heures sur Colab SERA interrompu : limite de
# session, deconnexion, onglet ferme. Sans point de reprise il faut tout
# recommencer.
_LAST = CKPT_DIR / "stage1_last.pth"
history, best, _start, _r = [], float("inf"), 0, None
if _LAST.exists():
    _r = torch.load(_LAST, map_location=DEVICE, weights_only=False)
    # Un checkpoint d'une AUTRE configuration ne decrit pas ce modele-ci.
    # Certains interrupteurs V8 changent les formes et feraient lever
    # load_state_dict ; d'autres non, et la reprise serait silencieusement
    # fausse. On compare donc la configuration, pas seulement les formes.
    _dif = None
    if _r.get("node_types") != NODE_TYPES:
        _dif = f"{len(_r.get('node_types') or [])} noeuds au lieu de {len(NODE_TYPES)}"
    elif _r.get("v8") != OmegaConf.to_container(V8):
        _dif = "interrupteurs V8 differents"
    if _dif:
        _vieux = _LAST.with_name(f"stage1_last.perime_{int(time.time())}.pth")
        _LAST.rename(_vieux)
        print(f"CHECKPOINT ECARTE : {_dif}")
        print(f"  conserve sous {_vieux.name} ; l'etage 1 repart de zero.")
        _r = None
if _r is not None:
    encoder.load_state_dict(_r["encoder_state_dict"])
    rcn_cell.load_state_dict(_r["rcn_cell_state_dict"])
    regression_head.load_state_dict(_r["regression_head_state_dict"])
    if bg_head is not None and "bg_head_state_dict" in _r:
        bg_head.load_state_dict(_r["bg_head_state_dict"])
    opt_s1.load_state_dict(_r["optimizer_state_dict"])
    history, best, _start = _r["history"], _r["best"], _r["epoch"] + 1
    print(f"REPRISE a l'epoque {_start + 1}/{EPOCHS_S1} "
          f"(meilleure val = {best:.5f})")

for ep in range(_start, EPOCHS_S1):
    t_ep = time.time()
    m = train_epoch_stage1(
        encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
        optimizer=opt_s1, data_loader=iterate_batches(train_dataset),
        device=DEVICE, epoch_idx=ep,
        lambda_reg=float(S1.lambda_reg), beta_rec=float(S1.beta_rec),
        gamma_dag_max=float(S1.gamma_dag_max),
        gamma_dag_warmup_epochs=int(S1.gamma_dag_warmup_epochs),
        lambda_l1=float(S1.lambda_l1),
        abort_on_collapse=bool(S1.abort_on_collapse),
        collapse_threshold=float(S1.collapse_threshold),
        dag_floor_projection=bool(S1.dag_floor_projection),
        dag_floor_min_norm=float(S1.dag_floor_min_norm),
        gradient_clipping=float(CONFIG.training.gradient_clipping),
        use_amp=(DEVICE.type == "cuda"),
        # --- A3 : la NLL Bernoulli-Gamma remplace la MSE ------------------
        bg_head=bg_head,
        # --- C7 : prior 3 niveaux, annele PAR NIVEAU ----------------------
        edge_prior=edge_prior, prior_node_order=NODE_TYPES,
        prior_level_floors=DEFAULT_LEVEL_FLOORS, prior_anneal_epochs=EPOCHS_S1,
        # --- exclusions imposees par bg_head (le loop les verifie) --------
        # Le seuil du LOOP prime sur celui de la tete : le laisser a 1,0
        # ferait ajuster la Gamma au-dessus de 1 mm/j, ce qui donne alpha=1,05
        # au lieu de 0,55 - la forme fausse d'un facteur 2, et c'est elle qui
        # gouverne les extremes.
        bg_wet_threshold=BG_WET_SEUIL,
        skip_block=skip_block, p1_tail_alpha=0.0, p3_k_samples=0,
        verbose=(ep == 0),
    )
    m["epoch"] = ep
    m["seconds"] = round(time.time() - t_ep, 1)
    history.append(m)
    print(f"[S1 {ep + 1:2d}/{EPOCHS_S1}] loss={m['loss']:.5f} "
          f"reg={m['loss_reg']:.5f} dag={m.get('loss_dag', 0.0):.4f} "
          f"({m['seconds']:.0f}s)")
    m["val_loss"] = validation_loss(val_dataset)
    print(f"          val={m['val_loss']:.5f}")

    _poids = {"encoder_state_dict": encoder.state_dict(),
              "rcn_cell_state_dict": rcn_cell.state_dict(),
              "regression_head_state_dict": regression_head.state_dict(),
              "node_types": NODE_TYPES, "v8": OmegaConf.to_container(V8)}
    if bg_head is not None:
        _poids["bg_head_state_dict"] = bg_head.state_dict()

    # A CHAQUE epoque : point de reprise, ETAT DE L'OPTIMISEUR compris. Sans
    # lui, reprendre repartirait avec des moments Adam nuls — ce ne serait pas
    # la meme trajectoire d'optimisation.
    torch.save({**_poids, "epoch": ep,
                "optimizer_state_dict": opt_s1.state_dict(),
                "history": history, "best": best}, _LAST)

    # Selection sur la VALIDATION, jamais sur l'entrainement.
    if m["val_loss"] < best:
        best = m["val_loss"]
        torch.save({"epoch": ep,
                    "encoder_state_dict": encoder.state_dict(),
                    "rcn_cell_state_dict": rcn_cell.state_dict(),
                    "regression_head_state_dict": regression_head.state_dict(),
                    **({"bg_head_state_dict": bg_head.state_dict()} if bg_head else {}),
                    "node_types": NODE_TYPES,
                    "v8": OmegaConf.to_container(V8)},
                   CKPT_DIR / "stage1_best.pth")

json.dump(history, open(RESULTS_DIR / "v8_stage1_history.json", "w"),
          indent=2, default=float)
# Trajectoire, en trois lignes. Sur une reprise ou la boucle ne tourne pas, la
# question "l'etage 1 a-t-il seulement appris ?" se poserait sinon sans reponse
# a l'ecran — et c'est la premiere a se poser quand mu_HR se revele inutilisable.
if history:
    _tr = [float(h["loss"]) for h in history]
    _va = [float(h["val_loss"]) for h in history if "val_loss" in h]
    print(f"trajectoire etage 1 sur {len(history)} epoques :")
    print(f"  entrainement {_tr[0]:.5f} -> {_tr[-1]:.5f} (min {min(_tr):.5f})")
    if _va:
        print(f"  validation   {_va[0]:.5f} -> {_va[-1]:.5f} (min {min(_va):.5f}"
              f" a l'epoque {_va.index(min(_va)) + 1})")
# CHARGER le meilleur checkpoint. Sans cela, les cellules suivantes
# travailleraient sur les poids de la DERNIERE epoque et le checkpoint
# "meilleur" ne serait qu'un fichier decoratif.
_ck = torch.load(CKPT_DIR / "stage1_best.pth", map_location=DEVICE, weights_only=False)
encoder.load_state_dict(_ck["encoder_state_dict"])
rcn_cell.load_state_dict(_ck["rcn_cell_state_dict"])
regression_head.load_state_dict(_ck["regression_head_state_dict"])
if bg_head is not None:
    bg_head.load_state_dict(_ck["bg_head_state_dict"])
print(f"meilleure perte de VALIDATION : {best:.5f} (epoque {_ck['epoch'] + 1}) "
      f"- poids recharges")

In [ ]:
# >>> Cell 6 : ce que le DAG a appris - decouvert contre impose
A_lag = rcn_cell.dag_matrix(masked=True).detach().cpu().numpy()
print(f"A(tau>=1) : norme={np.linalg.norm(A_lag):.4f} "
      f"asymetrie={np.abs(A_lag - A_lag.T).mean():.4f}")
if rcn_cell.A_inst is not None:
    A0 = rcn_cell.dag_matrix(masked=True, lag=0).detach().cpu().numpy()
    print(f"A(0)      : norme={np.linalg.norm(A0):.4f} "
          f"asymetrie={np.abs(A0 - A0.T).mean():.4f}")

def _score(A, lag_v, label):
    """Confronte UN operateur a la partie du prior qui lui revient.
    Melanger les lags rendrait le bilan faux sur 24 des 33 aretes.

    A LIRE AVEC PRUDENCE, trois limites que ce tableau ne leve pas :
    1. Le seuil est le 80e percentile de |A|, donc calibre pour retenir a peu
       pres autant d'aretes que le prior en contient. Comme l'entrainement
       tire les aretes du prior vers une cible non nulle pendant que L1 pousse
       le reste vers zero, cette separation reflete en partie la structure de
       la PENALITE, pas seulement celle des donnees. Un null par surrogates
       (comme T3) serait necessaire pour trancher.
    2. UNE SEULE graine. Le composant C8 du plan exige >=3 graines avant de
       declarer une arete retenue ou rejetee.
    3. Magnitude seule : une arete forte de SIGNE oppose au mecanisme
       physique compte ici comme "retenue".
    Une arete de niveau 1 rejetee signale plus probablement une difficulte
    d'optimisation qu'une decouverte negative : son plancher de prior est le
    plus eleve (0,08). Les niveaux 2 et 3 sont les seuls ou un rejet est
    reellement informatif."""
    P = edge_prior.matrix(lag=lag_v, node_order=NODE_TYPES)
    support = P != 0
    if not support.any():
        return None
    # Seuil sur le quantile 80 : on garde le meme budget d'aretes que le prior
    # plutot qu'un seuil absolu arbitraire qui dependrait de l'echelle de A.
    thr = float(np.quantile(np.abs(A), 0.80))
    found = np.abs(A) > thr
    kept, dropped = int((found & support).sum()), int((~found & support).sum())
    novel = int((found & ~support).sum())
    print(f"\nprior : {int(support.sum())} aretes | seuil |A| > {thr:.4f}")
    print(f"  retenues par les donnees : {kept}")
    print(f"  REJETEES                 : {dropped}   <- candidat signal de")
    print(f"                                            decouverte NEGATIF (C7),")
    print(f"                                            a confirmer sur >=3 graines")
    print(f"  hors prior               : {novel}   <- decouverte au-dela du prior")

    # Par niveau : le niveau 3 DOIT pouvoir tomber. C'est tout l'objet de
    # l'annealing par niveau - un prior speculatif que les donnees ne peuvent
    # pas rejeter n'est plus un prior, c'est une contrainte.
    print("\n  survie par niveau de credibilite :")
    per_level = {}
    for lvl, mask in edge_prior.level_masks(node_order=NODE_TYPES).items():
        mask = mask & support        # restreindre au lag de CET operateur
        if mask.any():
            r = float((found & mask).sum() / mask.sum())
            per_level[int(lvl)] = r
            print(f"    niveau {lvl} ({int(mask.sum()):2d} aretes) : {100 * r:4.0f} %")

    # Les aretes rejetees, nommees : c'est le livrable scientifique de C7.
    rejected = [(NODE_TYPES[i], NODE_TYPES[j])
                for i, j in zip(*np.where(~found & support))]
    if rejected:
        print("\n  aretes du prior REJETEES par les donnees :")
        for a, b in rejected:
            print(f"    {a} -> {b}")

    return {"operator": label, "n_prior": int(support.sum()), "kept": kept,
            "dropped": dropped, "novel": novel, "threshold": thr,
            "survival_by_level": per_level, "rejected_edges": rejected,
            "seeds": 1, "null_baseline": None,
            "avertissement": ("une seule graine, seuil au 80e percentile calibre "
                              "sur le budget du prior, magnitude seule (signe non "
                              "verifie) : resultat exploratoire, pas une "
                              "falsification")}


if edge_prior is not None:
    report = [r for r in (_score(A_lag, 1, "A(tau>=1)"),
                          (_score(A0, 0, "A(0)") if rcn_cell.A_inst is not None else None))
              if r is not None]
    json.dump(report, open(RESULTS_DIR / "v8_dag_vs_prior.json", "w"), indent=2)

In [ ]:
# >>> Cell 6b : ou meurt la dependance au jour ?
# mu_HR s'est revele sans correlation avec sa cible alors que la perte
# d'entrainement avait baisse de 45 %. Une perte qui descend sans que la
# validation bouge decrit un modele qui apprend la CLIMATOLOGIE puis memorise :
# un champ juste en moyenne, immobile d'un jour a l'autre.
#
# Reste a savoir OU la meteo se perd le long de la chaine. On mesure donc, a
# chaque etage, la part de variance portee par le JOUR plutot que par le lieu
# ou la variable. L'entree sert de temoin : c'est elle qui fixe combien il y
# avait a transmettre.
import itertools

_K_SONDE = 24
_ech = list(itertools.islice(train_dataset, 0, 10 * _K_SONDE, 10))
print(f"sonde sur {len(_ech)} jours espaces de 10 fenetres")

_etages = {"entree LR (temoin)": [], "etat RCN H_T": [],
           "features decodeur": [], "ancre E[log1p]": []}
_champs = []          # (ancre, HR vrai) par jour, pour la decomposition ci-dessous
for _m in STAGE1_MODULES:
    _m.eval()
with torch.no_grad():
    for _s in _ech:
        _b = convert_sample_v8(_s, builder, DEVICE)
        _lr = _b["lr"].to(DEVICE)
        _H0 = encoder.init_state(_b["hetero"])
        _sq = rcn_runner.run(_H0, [_lr[k] for k in range(_lr.shape[0])],
                             reconstruction_sources=None)
        _HT = _sq.states[-1]
        _f = regression_head(_HT, return_features=True)
        _etages["entree LR (temoin)"].append(_lr[-1].flatten().cpu())
        _etages["etat RCN H_T"].append(_HT.flatten().cpu())
        _etages["features decodeur"].append(_f.flatten().cpu())
        if bg_head is not None:
            _p, _a, _bb = bg_head(_f)
            _anc_j = BernoulliGammaHead.mean_log1p(_p, _a, _bb)
            _etages["ancre E[log1p]"].append(_anc_j.flatten().cpu())
            _tj = _b["residual"][-1].to(DEVICE)
            _blj = _b["baseline"][-1].to(DEVICE)
            if _tj.dim() == 3:
                _tj, _blj = _tj.unsqueeze(0), _blj.unsqueeze(0)
            _champs.append((_anc_j.squeeze().cpu(), (_blj + _tj).squeeze().cpu(),
                            _blj.squeeze().cpu()))
for _m in STAGE1_MODULES:
    _m.train()


def _part_jour(xs):
    """Fraction de la variance totale qui vient du jour."""
    if not xs:
        return float("nan")
    _X = torch.stack(xs).double()
    _v = float(_X.var())
    return float(_X.var(dim=0).mean() / _v) if _v > 1e-12 else float("nan")


_parts = {k: _part_jour(v) for k, v in _etages.items() if v}
print("part de variance portee par le jour :")
for _k, _v in _parts.items():
    print(f"  {_k:22s} {100 * _v:6.2f} %")

# Le temoin borne ce qui etait transmissible ; on cherche le premier etage qui
# effondre cette part. Un facteur 10 est deliberement grossier : il s'agit de
# reperer une rupture, pas de mesurer une attenuation.
_ref = _parts.get("entree LR (temoin)", float("nan"))
_mort = None
_prec = _ref
for _k, _v in list(_parts.items())[1:]:
    if _v == _v and _prec == _prec and _v < _prec / 10.0:
        _mort = _k
        break
    _prec = _v
_fin = list(_parts.values())[-1]
if _mort:
    print(f"RUPTURE : la dependance au jour s'effondre a l'etage << {_mort} >>.")
    print("  C'est la qu'il faut chercher, pas en aval.")
elif _fin == _fin and _ref == _ref and _fin < _ref / 10.0:
    # Angle mort du critere par etage : une chaine qui perd un facteur 3 a
    # chaque passage n'accuse aucun etage et n'en transmet pourtant rien.
    print(f"ATTENUATION CUMULATIVE : aucun etage ne casse seul, mais la sortie "
          f"ne garde que {100 * _fin / _ref:.1f} % de la part temporelle de "
          f"l'entree. Le defaut est reparti sur toute la chaine.")
elif _ref == _ref:
    print("Aucune rupture franche : la meteo traverse toute la chaine.")

# --- Niveau ou motif, et surtout : MIEUX QUE LA BASELINE ? -----------------
# Une part temporelle elevee dit que l'ancre BOUGE, pas qu'elle bouge bien. On
# separe donc le NIVEAU du jour (sa moyenne spatiale) du MOTIF (l'anomalie
# autour de ce niveau), et on correle chacun a la verite decomposee pareil.
#
# LE TEMOIN EST LA BASELINE, jamais zero. Correler l'ancre au HR vrai flatte :
# le HR ressemble deja beaucoup a la baseline (c'est son interpolation), donc
# recopier l'entree suffit a decrocher +0,9. Ce que l'etage 1 doit produire
# n'est pas HR, c'est HR MOINS la baseline — la part que l'interpolation ne
# donne pas. La ligne qui compte est donc la derniere : le residu.
if _champs:
    _AN = torch.stack([c[0] for c in _champs]).double()      # ancre  [K, H, W]
    _HR = torch.stack([c[1] for c in _champs]).double()      # verite
    _BL = torch.stack([c[2] for c in _champs]).double()      # baseline
    _ok = torch.isfinite(_HR).all(dim=0)                     # pixels valides partout
    _AN, _HR, _BL = _AN[:, _ok], _HR[:, _ok], _BL[:, _ok]    # [K, P]

    def _c(x, y):
        x, y = x.flatten() - x.mean(), y.flatten() - y.mean()
        _d = x.norm() * y.norm()
        return float(x @ y / _d) if _d > 1e-12 else float("nan")

    def _decomp(x):
        _n = x.mean(dim=1)
        return _n, x - _n[:, None]

    _na, _ma = _decomp(_AN)
    _nh, _mh = _decomp(_HR)
    _nb, _mb = _decomp(_BL)
    print("niveau du jour / motif spatial, correles a la verite :")
    print(f"  ancre     niveau {_c(_na, _nh):+.3f} ({float(_na.std()):.4f})  "
          f"motif {_c(_ma, _mh):+.3f} ({float(_ma.std()):.4f})")
    print(f"  BASELINE  niveau {_c(_nb, _nh):+.3f} ({float(_nb.std()):.4f})  "
          f"motif {_c(_mb, _mh):+.3f} ({float(_mb.std()):.4f})   <- le temoin")
    print(f"  verite                    ({float(_nh.std()):.4f})  "
          f"           ({float(_mh.std()):.4f})")

    # Ce que l'etage 2 recevra vraiment : mu = ancre - baseline contre
    # t = HR - baseline. C'est la meme quantite que la Cell 8, sur 24 jours.
    _mu, _t = _AN - _BL, _HR - _BL
    _r2 = 1.0 - float((_t - _mu).pow(2).sum() / _t.pow(2).sum())
    print(f"  RESIDU    corr(mu, t) = {_c(_mu, _t):+.3f} | R2 = {_r2:+.3f} "
          f"| sigma {float(_mu.std()):.4f} contre {float(_t.std()):.4f}")
    if _c(_ma, _mh) <= _c(_mb, _mh) + 0.02:
        print("  -> l'ancre ne fait PAS mieux que la baseline sur le motif : "
              "l'etage 1 recopie son entree au lieu d'ajouter du fin.")
    elif float(_ma.std()) < 0.1 * float(_mh.std()):
        print("  -> la tete ne produit quasiment PAS de motif spatial.")
    del _AN, _HR, _BL, _ma, _mh, _mb, _mu, _t
del _etages, _ech, _champs

In [ ]:
# >>> Cell 7 : A1 - calibration de Jensen + audit de conformite de l'etage 1
from st_cdgm.evaluation.jensen import JensenCorrector
from st_cdgm.training.stage1_paths import predict_mu_hr

for _m in STAGE1_MODULES:
    _m.eval()

@torch.no_grad()
def collect(ds, n_max=400):
    """mu / cible / baseline en espace log1p, [N, H, W]."""
    mus, tgs, bls = [], [], []
    for i, s in enumerate(ds):
        if i >= n_max:
            break
        b = convert_sample_v8(s, builder, DEVICE)
        t = b["residual"][-1].to(DEVICE)
        if t.dim() == 3:
            t = t.unsqueeze(0)
        bl = b["baseline"][-1].to(DEVICE)
        if bl.dim() == 3:
            bl = bl.unsqueeze(0)
        mu = predict_mu_hr(b, variant="causal", encoder=encoder, rcn_runner=rcn_runner,
                           regression_head=regression_head, builder=builder,
                           device=DEVICE, target_shape=t.shape[-2:], bg_head=bg_head)
        mus.append(mu.reshape(t.shape[-2:]).cpu())
        tgs.append(t.reshape(t.shape[-2:]).cpu())
        bls.append(bl.reshape(t.shape[-2:]).cpu())
    return (torch.stack(mus).numpy(), torch.stack(tgs).numpy(), torch.stack(bls).numpy())

mu_tr, tg_tr, bl_tr = collect(train_dataset)
print("echantillon de calibration :", mu_tr.shape)

# s^2 calibre sur le TRAIN uniquement. C'est un parametre de calibration :
# l'estimer sur le test ferait fuiter la cible dans la metrique.
jc = JensenCorrector.fit(mu_tr, tg_tr, bl_tr) if V8.jensen else None
if jc is not None:
    print(jc)
    torch.save(jc.state_dict(), CKPT_DIR / "jensen.pt")

# --- Audit de conformite. C1 juge le biais APRES correction de Jensen :
# sinon il mesurerait l'inegalite de Jensen et non le modele. Un etage 1
# EXACT en log1p affiche ~22 % de biais conditionnel sans la correction.
mu_te, tg_te, bl_te = collect(test_dataset, n_max=N_EVAL)
x_mm = np.expm1(np.clip(bl_te + tg_te, -20, 20))

def cond_bias(pred_mm, min_par_bin=500):
    """Biais relatif par decile de la PREDICTION, pas de la verite.

    C'est une courbe de fiabilite : le biais conditionne a l'intensite PREVUE.
    A ne pas lire comme un biais conditionne a l'intensite observee - les deux
    divergent sur un domaine a fort gradient orographique.
    """
    q = np.unique(np.quantile(pred_mm, np.linspace(0, 1, 11)))
    if q.size < 2:
        return []                    # champ constant : aucun bin exploitable
    bid = np.clip(np.digitize(pred_mm.ravel(), q[1:-1]), 0, len(q) - 2)
    xr_, mr_ = x_mm.ravel(), pred_mm.ravel()
    return [100 * (xr_[bid == k].mean() - mr_[bid == k].mean())
            / max(xr_[bid == k].mean(), 1e-9)
            for k in range(len(q) - 1) if (bid == k).sum() > min_par_bin]


naive_mm = np.expm1(np.clip(bl_te + mu_te, -20, 20))
corr_mm = jc.to_mm(mu_te, bl_te, delta=0.0).numpy() if jc is not None else naive_mm
_bn, _bc = cond_bias(naive_mm), cond_bias(corr_mm)
print()

if not _bn or not _bc:
    # Aucun bin exploitable = champ predit quasi CONSTANT. C'est un RESULTAT,
    # pas un plantage : soit l'echantillon est trop petit, soit l'etage 1
    # s'est effondre sur une prediction plate. Ce second cas est un mode de
    # defaillance documente de cette architecture - la cellule d'audit doit le
    # SIGNALER, pas lever une exception qui le masque.
    b_naive = b_corr = float("nan")
    _cause = "echantillon trop petit" if len(mu_te) < 50 else "ETAGE 1 PLAT"
    print(f"C1 NON CALCULABLE : moins de 2 deciles distincts de plus de 500 "
          f"pixels sur {len(mu_te)} echantillons. Ecart-type de la prediction "
          f"= {float(np.std(naive_mm)):.4f} mm/j  ->  {_cause}")
else:
    b_naive, b_corr = max(map(abs, _bn)), max(map(abs, _bc))
    print(f"biais conditionnel max : naif {b_naive:5.1f} %  ->  "
          f"corrige {b_corr:5.1f} %")
    print(f"C1 {'PASS' if b_corr < 5.0 else 'FAIL'} (seuil 5 %) | "
          f"part imputable a Jensen : {b_naive - b_corr:.1f} points")

json.dump({"C1_bias_corrected_pct": float(b_corr),
           "C1_bias_naive_pct": float(b_naive),
           "C1_jensen_share_pct": float(b_naive - b_corr),
           "C1_pass": (bool(b_corr < 5.0) if b_corr == b_corr else None),
           "C1_n_bins": len(_bc)},
          open(RESULTS_DIR / "v8_stage1_audit.json", "w"), indent=2)

In [ ]:
# >>> Cell 8 : gel de l'etage 1 + cache pour l'etage 2
from st_cdgm.training.two_stage import freeze_stage1, precompute_stage1_outputs

freeze_stage1(encoder, rcn_cell, regression_head, *([bg_head] if bg_head else []))
rcn_cell.A_dag.requires_grad_(False)
if rcn_cell.A_inst is not None:
    rcn_cell.A_inst.requires_grad_(False)
print("etage 1 gele, A_dag et A(0) compris - le DAG devient une feature OOD assumee")

# L'ancre mise en cache est mu = p*alpha*beta reexprimee en residu log1p,
# pas la projection 1 canal du decodeur (qui n'est plus entrainee sous A3).
cache = precompute_stage1_outputs(
    encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
    train_dataset=train_dataset,
    iterate_batches_fn=lambda s: convert_sample_v8(s, builder, DEVICE),
    device=DEVICE, bg_head=bg_head)
print({k: tuple(v.shape) for k, v in cache.items()})

# sigma_data APRES le cache, et mesure sur la VRAIE cible de la diffusion.
# calibrate_sigma_data_variant n'accepte pas bg_head : elle passerait par
# regression_head(H_T), donc par upsample[-1] - la projection 1 canal que la
# tete BG court-circuite et qui n'a JAMAIS recu de gradient. sigma_data aurait
# ete l'ecart-type d'une projection aleatoire, et il alimente c_skip/c_out/c_in
# du preconditionneur EDM.
_delta = cache["delta_target"]
_mask = cache["valid_mask"].bool()
SIGMA_DATA = float(_delta[_mask].std())
print(f"sigma_data = {SIGMA_DATA:.5f}  (ecart-type du residu reellement diffuse)")

# --- CALIBRATION DE mu_HR : le controle qui doit tomber AVANT l'etage 2 -----
# delta_target = t - mu, ou t = HR_log - baseline_log est le residu VRAI. Si mu
# sur-estime l'amplitude de t, la diffusion passe sa capacite a ANNULER mu au
# lieu d'ajouter du detail — c'est le risque n°1 pre-enregistre (V6' prereg,
# `risk_1_and_contingency`), avec pour declencheur corr(D_y, mu) < -0,3.
# Ce diagnostic-ci est cote DONNEES : il se lit sur le cache, sans entrainer,
# donc avant de payer les heures de l'etage 2.
_pas = max(1, cache["mu_HR"].shape[0] // 700)      # ~700 fenetres suffisent
_m = cache["valid_mask"][::_pas].bool()
_mu_c = cache["mu_HR"][::_pas][_m].double()
_dl_c = cache["delta_target"][::_pas][_m].double()
_t_c = _dl_c + _mu_c
_sig_t, _sig_mu = float(_t_c.std()), float(_mu_c.std())
_corr = lambda a, b: float(((a - a.mean()) @ (b - b.mean()))
                           / ((a - a.mean()).norm() * (b - b.mean()).norm()))
# a_opt : le facteur d'echelle qui MINIMISE ||t - a*mu||. Loin de 1 = mu mal
# calibre en amplitude, ce qui est un defaut different d'un mu peu informatif.
_a_opt = float((_t_c @ _mu_c) / (_mu_c @ _mu_c))
# R2 de mu comme predicteur de t. NEGATIF = mu fait pire que ne rien predire :
# l'etage 2 devrait alors commencer par defaire l'etage 1.
_r2 = 1.0 - float((_t_c - _mu_c).pow(2).sum() / _t_c.pow(2).sum())
print(f"calibration mu_HR : sigma(t)={_sig_t:.4f} sigma(mu)={_sig_mu:.4f} "
      f"(rapport {_sig_mu / max(_sig_t, 1e-9):.2f})")
print(f"                    corr(t,mu)={_corr(_t_c, _mu_c):+.3f} | "
      f"a_opt={_a_opt:.3f} | R2={_r2:+.3f}")
print(f"                    corr(delta,mu)={_corr(_dl_c, _mu_c):+.3f} "
      f"(seuil pre-enregistre : -0,30)")

# Si mu est mauvais, DEUX pannes differentes le produisent et elles n'ont pas
# le meme remede : soit l'ancre ne bouge pas d'un jour a l'autre (le decodeur a
# appris la climatologie, le relief — la meteo n'arrive pas jusqu'a lui), soit
# elle ne bouge nulle part (la tete est restee a son initialisation). On mesure
# la part de variance PORTEE PAR LE TEMPS, avec le HR vrai comme temoin.
_va = cache["valid_mask"][::_pas].bool()
_pix = _va.all(dim=0)                       # pixels valides sur tout l'echantillon
_anc = (cache["mu_HR"][::_pas] + cache["baseline_log"][::_pas])[:, _pix].double()
_hrv = (cache["baseline_log"][::_pas] + cache["delta_target"][::_pas]
        + cache["mu_HR"][::_pas])[:, _pix].double()


def _part_temporelle(x):
    """Fraction de la variance totale qui vient du jour, pas du lieu."""
    v = float(x.var())
    return float(x.var(dim=0).mean() / v) if v > 1e-12 else float("nan")


_pt_anc, _pt_hr = _part_temporelle(_anc), _part_temporelle(_hrv)
print(f"                    part temporelle : ancre {100 * _pt_anc:.1f} % | "
      f"HR vrai {100 * _pt_hr:.1f} % | ecart-type de l'ancre {float(_anc.std()):.4f}")
_diagnostic = ""
if float(_anc.std()) < 0.02:
    _diagnostic = ("l'ancre est quasi CONSTANTE partout : la tete BG est restee "
                   "a son initialisation, rien ne l'a entrainee.")
elif _pt_anc < 0.1 * _pt_hr:
    _diagnostic = ("l'ancre ne varie pas d'un jour a l'autre : le decodeur a "
                   "appris un champ STATIQUE (climatologie/relief) et l'etat du "
                   "RCN n'apporte pas la meteo.")
if _diagnostic:
    print(f"                    -> {_diagnostic}")
del _anc, _hrv, _va, _pix
if _r2 < 0.0 and not os.environ.get("V8_IGNORE_MU_CALIB"):
    raise RuntimeError(
        f"R2(mu_HR) = {_r2:+.3f} < 0 : l'etage 1 predit PIRE que zero, la "
        f"diffusion devrait d'abord defaire mu (a_opt={_a_opt:.3f}, "
        f"sigma(mu)/sigma(t)={_sig_mu / max(_sig_t, 1e-9):.2f}). "
        + (f"DIAGNOSTIC : {_diagnostic} " if _diagnostic else "")
        + f"Entrainer "
        f"l'etage 2 par-dessus coute des heures pour un resultat ininterpretable. "
        f"Reprendre l'etage 1, ou basculer sur la variante pre-enregistree "
        f"V6'.1 (delta = HR - baseline, mu en conditionnement seul). "
        f"V8_IGNORE_MU_CALIB=1 pour passer outre en connaissance de cause.")
if _corr(_dl_c, _mu_c) < -0.3:
    print("ATTENTION : declencheur pre-enregistre V6'.1 ATTEINT sur les donnees.")
del _mu_c, _dl_c, _t_c, _m

torch.save({"sigma_data": SIGMA_DATA, "node_types": NODE_TYPES,
            "v8": OmegaConf.to_container(V8)},
           CKPT_DIR / "stage1_frozen_meta.pth")

In [ ]:
# >>> Cell 9 : etage 2 - diffusion EDM sur le residu
from torch.utils.data import TensorDataset, DataLoader
from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from st_cdgm.training.two_stage import train_epoch_stage2_cached

torch.manual_seed(SEED)
S2 = CONFIG.two_stage.stage2
# sigma_data ne se passe PAS au constructeur : il vit dans l'EDMConfig. On part
# du bloc `diffusion.edm` du YAML et on y injecte la valeur CALIBREE en Cell 8,
# sinon le preconditionneur travaillerait avec la valeur figee du fichier
# (0,1) alors que l'echelle reelle du residu est mesuree a l'execution.
_edm = dict(CONFIG.diffusion.get("edm", {}))
_edm["sigma_data"] = SIGMA_DATA
diffusion = CausalDiffusionDecoder(
    in_channels=1,
    conditioning_dim=int(CONFIG.diffusion.conditioning_dim),
    height=int(CONFIG.diffusion.height), width=int(CONFIG.diffusion.width),
    unet_kwargs=OmegaConf.to_container(CONFIG.diffusion.unet_kwargs, resolve=True),
    scheduler_type=str(CONFIG.diffusion.scheduler_type),
    edm_config=EDMConfig.from_yaml_dict(_edm),
    # causal_concat=True est OBLIGATOIRE : sans lui l'UNet est bati avec 1 seul
    # canal d'entree et compute_loss_edm(mu_HR=..., baseline_log=...) leve.
    causal_concat=True,
    conv_padding_mode=str(CONFIG.diffusion.get("conv_padding_mode", "zeros")),
    anti_checkerboard=bool(CONFIG.diffusion.get("anti_checkerboard", False)),
    # Requis a batch=64 : recompute les activations au backward, ~30 % plus lent
    # et ~50 % de VRAM en moins. Sans lui, OOM sur T4 16 Go.
    use_gradient_checkpointing=True,
).to(DEVICE)
print(f"parametres etage 2 : {sum(p.numel() for p in diffusion.parameters()):,}")

cached = TensorDataset(cache["mu_HR"], cache["baseline_log"],
                       cache["delta_target"], cache["valid_mask"])
# CONFIG.training.batch_size = 64 vise l'A100 80 Go : le YAML de reference dit
# lui-meme que 64 sur cet UNet "OOM meme sur A100" sans recompute d'activations.
# Sur T4 16 Go on descend a 32. Cela ne change PAS le budget d'entrainement :
# une epoque reste une passe sur le cache, donc le nombre de tirages est le
# meme — seul le nombre de pas double.
_VRAM = (torch.cuda.get_device_properties(0).total_memory / 2 ** 30
         if DEVICE.type == "cuda" else 0.0)
# Seuil a 30 GiB : une A100-40 rapporte 39,4 GiB, un seuil a 40 la classait
# donc avec les L4 et divisait son debit par deux pour rien.
BS = int(os.environ.get("V8_BS_S2", 0)) or (64 if _VRAM > 30 else 32)
print(f"batch etage 2 : {BS} ({_VRAM:.0f} GiB de VRAM)")
# drop_last=True : si le cache contient MOINS que batch_size, le DataLoader rend
# zero batch et train_epoch_stage2_cached tourne a vide en renvoyant n_batches=0
# sans lever. Observe sur un smoke : l'etage 2 n'avait rien entraine et rien ne
# le disait. On refuse plutot que d'entrainer dans le vide.
if len(cached) < BS:
    raise ValueError(
        f"cache de {len(cached)} echantillons pour un batch de {BS} : avec "
        f"drop_last=True le loader serait VIDE et l'etage 2 ne s'entrainerait "
        f"pas. Baisser V8_BS_S2 ou augmenter le jeu.")

def cached_loader():
    for mu, bl, dt, vm in DataLoader(cached, batch_size=BS, shuffle=True, drop_last=True):
        yield {"mu_HR": mu.to(DEVICE), "baseline_log": bl.to(DEVICE),
               "delta_target": dt.to(DEVICE), "valid_mask": vm.to(DEVICE)}


# Nombre d'epoques DERIVE du budget en tirages. Une epoque = une passe sur le
# cache, donc EPOCHS = tirages_vises / taille_du_cache — independamment du
# batch et du stride. C'est cette derivation qui empeche de refaire l'erreur
# d'origine : une constante d'epoques ne veut rien dire tant que la taille du
# cache peut bouger sous elle.
EPOCHS_S2 = int(os.environ.get("V8_EPOCHS_S2", 0)) or max(
    1, -(-TARGET_DRAWS_S2 // len(cached)))
print(f"budget etage 2 : {TARGET_DRAWS_S2:,} tirages / {len(cached)} fenetres "
      f"= {EPOCHS_S2} epoques de {len(cached) // BS} pas")

opt_s2 = torch.optim.AdamW(diffusion.parameters(), lr=float(S2.lr),
                           betas=(0.9, 0.99))

# EMA des poids (V5 Track B1, convention Karras EDM2). V5 et CorrDiff evaluent
# tous les deux la moyenne mobile, pas les poids du dernier pas ; s'en passer
# ici desavantagerait V8 sur un facteur qui n'a rien d'architectural.
# decay=0.999 et non 0.9999 : le shadow n'est PAS mis a jour pendant le warmup,
# il repart donc de l'init ALEATOIRE, dont le poids residuel apres n pas vaut
# decay^n. A 0,9999 sur 20 000 pas il resterait ~13 % d'init aleatoire dans les
# poids evalues ; a 0,999 il n'en reste rien des le dixieme de ce budget.
# Horizon d'averaging : ~1 000 pas.
import copy
ema = copy.deepcopy(diffusion)
EMA_DECAY = 0.999

# Programme de pas : warmup lineaire puis cosinus (Karras EDM §5, comme V5).
# Sur ~21 000 pas un lr constant laisse le modele osciller autour du minimum en
# fin de course ; le warmup evite que les premiers pas, sur des poids
# aleatoires, ne detruisent l'echelle de l'UNet.
_WARM = max(1, min(5, EPOCHS_S2 // 50))
sched_s2 = torch.optim.lr_scheduler.SequentialLR(
    opt_s2,
    [torch.optim.lr_scheduler.LinearLR(opt_s2, start_factor=0.1,
                                       total_iters=_WARM),
     torch.optim.lr_scheduler.CosineAnnealingLR(
         opt_s2, T_max=max(1, EPOCHS_S2 - _WARM), eta_min=float(S2.lr) / 50)],
    milestones=[_WARM])
print(f"etage 2 : lr={float(S2.lr):.1e} | sigma_data={SIGMA_DATA:.5f} | "
      f"EMA decay={EMA_DECAY} | warmup {_WARM} ep puis cosinus")

_LAST2 = CKPT_DIR / "stage2_last.pth"
_ARCH_S2 = list(CONFIG.diffusion.unet_kwargs.block_out_channels)


def _incompatible(r):
    """Pourquoi ce checkpoint ne peut PAS servir a reprendre — ou None.

    Verifie AVANT tout load_state_dict : `strict=True` copie les tenseurs qui
    correspondent avant de lever sur les autres, et laisserait donc le modele
    a moitie ecrase par les poids d'un run etranger.
    """
    _a = r.get("unet_block_out_channels")
    if _a is None:                       # checkpoint anterieur a ce champ
        _w = r.get("diffusion_state_dict", {}).get("unet.conv_in.weight")
        _a = [int(_w.shape[0])] if _w is not None else None
    if _a is not None and list(_a)[:1] != _ARCH_S2[:1]:
        return f"UNet {list(_a)} au lieu de {_ARCH_S2}"
    # sigma_data alimente c_skip/c_out/c_in du preconditionneur EDM : reprendre
    # avec une autre valeur entrainerait un modele autrement preconditionne,
    # sans que rien ne le signale.
    _s = r.get("sigma_data")
    if _s is not None and abs(float(_s) - SIGMA_DATA) > 0.05 * SIGMA_DATA:
        return f"sigma_data {float(_s):.5f} au lieu de {SIGMA_DATA:.5f}"
    return None


hist2, _start2, _r2 = [], 0, None
if _LAST2.exists():
    _r2 = torch.load(_LAST2, map_location=DEVICE, weights_only=False)
    _pourquoi = _incompatible(_r2)
    if _pourquoi:
        # Le checkpoint vient d'une AUTRE configuration : le charger leverait
        # un mur de "size mismatch". On l'ecarte sans le detruire et on repart
        # de zero — c'est la seule chose correcte, l'entrainement precedent ne
        # decrit pas ce modele-ci.
        _vieux = _LAST2.with_name(f"stage2_last.perime_{int(time.time())}.pth")
        _LAST2.rename(_vieux)
        print(f"CHECKPOINT ECARTE : {_pourquoi}")
        print(f"  conserve sous {_vieux.name} ; l'etage 2 repart de zero.")
        _r2 = None
if _r2 is not None:
    diffusion.load_state_dict(_r2["diffusion_state_dict"])
    # ORDRE NON NEGOCIABLE : l'optimiseur AVANT le scheduler. `SequentialLR`
    # remet le lr a sa valeur de warmup a la construction, et son
    # `load_state_dict` ne restaure que des compteurs — c'est le state_dict de
    # l'OPTIMISEUR qui porte le lr courant. Inverser les deux lignes ferait
    # reprendre une epoque a 2e-5 au lieu de 1,3e-4, sans rien signaler.
    if "optimizer_state_dict" in _r2:
        opt_s2.load_state_dict(_r2["optimizer_state_dict"])
    if "ema_state_dict" in _r2:
        ema.load_state_dict(_r2["ema_state_dict"])
    hist2, _start2 = _r2.get("history", []), _r2["epoch"] + 1
    if "scheduler_state_dict" in _r2:
        sched_s2.load_state_dict(_r2["scheduler_state_dict"])
    else:
        # Checkpoint anterieur au programme de pas : le rejouer a vide, sinon
        # la reprise repartirait au lr du debut au lieu du lr courant.
        for _ in range(_start2):
            sched_s2.step()
    print(f"REPRISE etage 2 a l'epoque {_start2 + 1}/{EPOCHS_S2} "
          f"(lr={opt_s2.param_groups[0]['lr']:.2e})")

def _epoque(ep):
    return train_epoch_stage2_cached(
        diffusion_decoder=diffusion, optimizer=opt_s2,
        cached_dataloader=cached_loader(), device=DEVICE,
        use_amp=(DEVICE.type == "cuda"), gradient_clipping=1.0,
        # conditioning dropout : entraine la branche inconditionnelle, requise
        # pour cfg_scale > 1 a l'inference (CorrDiff, Mardani 2024 sec 4.2).
        conditioning_dropout_prob=float(
            CONFIG.diffusion.get("conditioning_dropout_prob", 0.13)),
        ema_model=ema, ema_decay=EMA_DECAY, ema_warmup_steps=0,
        verbose=(ep == _start2))


for ep in range(_start2, EPOCHS_S2):
    t_ep = time.time()
    while True:
        try:
            m2 = _epoque(ep)
            break
        except torch.cuda.OutOfMemoryError:
            # L'UNet de reference tient sur T4 a batch 32 d'apres l'estimation,
            # pas d'apres une mesure. Si elle est fausse, l'OOM tombe apres
            # l'etage 1 — plusieurs heures deja payees. On divise le batch et
            # on refait l'epoque plutot que de perdre le run. Le budget en
            # tirages ne bouge pas ; seul le nombre de pas augmente.
            if BS <= 4:
                raise
            torch.cuda.empty_cache()
            BS //= 2
            print(f"  OOM -> batch reduit a {BS}, epoque refaite")
    m2 = dict(m2); m2["epoch"] = ep; m2["seconds"] = round(time.time() - t_ep, 1)
    m2["batch_size"] = BS
    m2["lr"] = opt_s2.param_groups[0]["lr"]
    sched_s2.step()
    hist2.append(m2)
    print(f"[S2 {ep + 1:2d}/{EPOCHS_S2}] {m2}")
    if int(m2.get("n_batches", 0)) == 0:
        raise RuntimeError("epoque etage 2 sans aucun batch : rien n'a ete "
                           "entraine, verifier la taille du cache.")
    if ep == _start2:
        # Le budget se lit en PAS, et le temps total se mesure a la premiere
        # epoque plutot que de se decouvrir a la huitieme heure.
        _nb = int(m2["n_batches"])
        print(f"  -> {_nb} batches/epoque, {_nb * EPOCHS_S2:,} pas au total, "
              f"~{m2['seconds'] * (EPOCHS_S2 - _start2) / 3600:.1f} h restantes "
              f"(interruptible : la reprise est en place)")
    torch.save({"epoch": ep, "diffusion_state_dict": diffusion.state_dict(),
                "optimizer_state_dict": opt_s2.state_dict(),
                "scheduler_state_dict": sched_s2.state_dict(),
                "ema_state_dict": ema.state_dict(),
                # Signature d'architecture : sans elle, un checkpoint d'un run
                # a l'UNet different se charge (mal) au lieu d'etre ecarte.
                "unet_block_out_channels": _ARCH_S2,
                "sigma_data": SIGMA_DATA, "history": hist2}, _LAST2)

# L'evaluation porte sur les poids EMA, comme V5 et CorrDiff. Les poids vifs
# restent dans le checkpoint : une reprise repart de la vraie trajectoire.
diffusion.load_state_dict(ema.state_dict())
print("poids EMA charges pour l'evaluation")

json.dump(hist2, open(RESULTS_DIR / "v8_stage2_history.json", "w"),
          indent=2, default=float)

In [ ]:
# >>> Cell 10 : evaluation IN-PROTOCOL (celle du 3-way V6' / ORACLE / CorrDiff)
# Le but de cette cellule n'est PAS de produire "des metriques" mais de produire
# les MEMES metriques, dans les MEMES conditions, que celles deja mesurees sur
# les autres modeles — sinon la Cell 11 comparerait des protocoles, pas des
# modeles. Tout ce qui suit est donc contraint :
#   evaluate_ensemble        la meme fonction (composition mm PAR MEMBRE)
#   K=32, 24 pas, cfg 0.0    les reglages du 3-way
#   graines 1000+k           les memes tirages
#   split de test complet     au meme stride
#   clim per-pixel partagee  le meme .npz quand il est disponible
import time as _time
from st_cdgm.evaluation.eval_metrics_dual_convention import (
    evaluate_ensemble, to_mm_day)
from st_cdgm.evaluation.two_stage_inference import sample_once_edm

diffusion.eval()
METRICS_PATH = RESULTS_DIR / "v8_metrics_inprotocol.json"

# --- 1. etage 1 sur TOUT le split de test ---------------------------------
# Meme chemin que le cache d'entrainement (Cell 8), tete BG comprise : mu doit
# designer la meme quantite des deux cotes, sinon delta n'a pas le meme sens.
test_cache = precompute_stage1_outputs(
    encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
    train_dataset=test_dataset,
    iterate_batches_fn=lambda s: convert_sample_v8(s, builder, DEVICE),
    device=DEVICE, bg_head=bg_head)
mu_all, base_all = test_cache["mu_HR"], test_cache["baseline_log"]
delta_all = test_cache["delta_target"]
N_TEST = int(mu_all.shape[0])
print(f"test : {N_TEST} fenetres (split complet, stride {STRIDE_EVAL})")

# --- 2. climatologie per-pixel (Convention A, ETCCDI) ---------------------
# Reutiliser le .npz des runs precedents quand il existe : un seuil p99 estime
# sur un echantillon different donne un F1 different a modele EGAL. Les clefs
# `clim_p95`/`clim_p99` sont celles ecrites par les runs 9-node et V6'.
_cands = [RESULTS_DIR / "clim_p95_p99.npz"]
if IN_COLAB:
    _cands = [Path(DRIVE_ROOT) / "oracle_9node/seed_42/phase8/clim_p95_p99.npz",
              Path(DRIVE_ROOT) / "ckpt_v2_corrdiff_normal/clim_p95_p99.npz",
              Path(DRIVE_ROOT) / "oracle_v6_prime/seed_42/clim_p95_p99.npz"] + _cands
_cp = next((p for p in _cands if p.exists()), None)
if _cp is not None:
    _z = np.load(_cp)
    clim95 = torch.from_numpy(_z["clim_p95"].astype("float32"))
    clim99 = torch.from_numpy(_z["clim_p99"].astype("float32"))
    CLIM_SOURCE = str(_cp)
else:
    # Recompose le HR VRAI depuis le cache d'ENTRAINEMENT : baseline + mu +
    # delta_target redonne exactement la cible, independamment du modele. C'est
    # la periode d'entrainement qui sert de reference (standard ETCCDI) — la
    # calculer sur la verite de test definirait l'evenement extreme a partir des
    # echantillons servant a compter les succes.
    _hr_mm = to_mm_day(cache["baseline_log"] + cache["mu_HR"]
                       + cache["delta_target"]).squeeze(1).numpy()
    _p95 = np.nanpercentile(_hr_mm, 95.0, axis=0).astype("float32")
    _p99 = np.nanpercentile(_hr_mm, 99.0, axis=0).astype("float32")
    np.savez(RESULTS_DIR / "clim_p95_p99.npz", clim_p95=_p95, clim_p99=_p99)
    clim95, clim99 = torch.from_numpy(_p95), torch.from_numpy(_p99)
    CLIM_SOURCE = f"calculee sur {_hr_mm.shape[0]} jours d'entrainement"
    del _hr_mm
print(f"climatologie : {CLIM_SOURCE}")
print(f"  p95 med={float(clim95.median()):.2f} | p99 med={float(clim99.median()):.2f} mm/j")

# --- 3. echantillonnage par lots ------------------------------------------
# Un membre a la fois sur TOUT le split, par paquets de EVAL_BATCH. Echantillonner
# sample par sample (batch 1) multiplierait le nombre de forwards UNet par
# EVAL_BATCH : plusieurs heures au lieu de dizaines de minutes.
_EPOCHS_DONE = len(hist2)


def sample_ensemble(K):
    members, t0 = [], _time.time()
    with torch.no_grad():
        for k in range(K):
            torch.manual_seed(1000 + k)   # memes graines que le 3-way
            chunks = []
            for i0 in range(0, N_TEST, EVAL_BATCH):
                sl = slice(i0, min(i0 + EVAL_BATCH, N_TEST))
                chunks.append(sample_once_edm(
                    diffusion, mu_HR=mu_all[sl].to(DEVICE),
                    baseline_log=base_all[sl].to(DEVICE),
                    scheduler_type="edm_karras", num_steps=NUM_STEPS,
                    cfg_scale=CFG_SCALE).cpu())
            members.append(torch.cat(chunks, 0))
            if k == 0 or (k + 1) % 4 == 0:
                _el = _time.time() - t0
                print(f"  membre {k + 1}/{K} | {_el:.0f}s ecoule | "
                      f"ETA {_el / (k + 1) * (K - k - 1):.0f}s", flush=True)
    return torch.stack(members, 0)          # [K, N, 1, H, W] sur CPU


# PERSISTANCE. L'echantillonnage coute des dizaines de minutes ; relancer la
# cellule pour lire la table de la Cell 11 ne doit pas le refaire. On invalide
# quand meme le cache si l'etage 2 a avance depuis : sinon la table afficherait
# en silence les metriques d'un modele moins entraine.
_prev = json.load(open(METRICS_PATH)) if METRICS_PATH.exists() else None
if _prev and _prev.get("protocol", {}).get("stage2_epochs") == _EPOCHS_DONE \
        and _prev.get("protocol", {}).get("K") == K_VERDICT:
    res = _prev["metrics"]
    print(f"metriques relues ({METRICS_PATH}) - supprimer le fichier pour recalculer")
else:
    if _prev:
        print(f"cache de metriques perime (etage 2 : "
              f"{_prev.get('protocol', {}).get('stage2_epochs')} -> {_EPOCHS_DONE} "
              f"epoques) - recalcul")
    ens = sample_ensemble(K_VERDICT)
    try:
        res = evaluate_ensemble(ens.to(DEVICE), mu_all.to(DEVICE),
                                base_all.to(DEVICE), delta_all.to(DEVICE),
                                clim99.to(DEVICE), clim95.to(DEVICE))
    except torch.cuda.OutOfMemoryError:
        # L'ensemble fait ~0,7 Go et evaluate_ensemble en materialise deux
        # copies. Plutot que de perdre l'echantillonnage sur un OOM a la
        # derniere ligne, on refait le calcul sur CPU : plus lent, identique.
        torch.cuda.empty_cache()
        print("OOM GPU sur les metriques -> recalcul sur CPU")
        res = evaluate_ensemble(ens, mu_all, base_all, delta_all,
                                clim99, clim95)
    del ens
    json.dump({"metrics": res,
               "protocol": {"K": K_VERDICT, "num_steps": NUM_STEPS,
                            "cfg_scale": CFG_SCALE, "n_test": N_TEST,
                            "stride": int(STRIDE_EVAL), "gcm": "ACCESS-CM2",
                            "clim_source": CLIM_SOURCE,
                            "stage2_epochs": _EPOCHS_DONE,
                            "composition": "mm par membre (evaluate_ensemble)"},
               "v8_nominal": OmegaConf.to_container(V8)},
              open(METRICS_PATH, "w"), indent=2, default=float)
    print(f"ecrit : {METRICS_PATH}")

print()
for _k in ("conv_A_F1p99", "conv_B_F1p99", "rmse", "pearson_global", "crps"):
    print(f"  {_k:16s} = {res[_k]:.4f}")

In [ ]:
# >>> Cell 11 : comparaison aux modeles deja evalues + verdict pre-enregistre
# Les autres modeles ne sont PAS reevalues : on relit la table produite par le
# 3-way V6' (metrique Conv A corrigee, meme split, meme K, meme composition mm).
# Cette cellule ne vaut que si le protocole coincide — elle le verifie et le dit
# plutot que d'aligner des nombres incomparables.
TABLE = ["conv_A_F1p99", "conv_A_F1p95", "conv_B_F1p99", "conv_B_F1p95",
         "pearson_global", "rmse", "mae", "crps", "ssr", "fss_p99",
         "rapsd_distance", "bias_mm", "rx1day_bias", "qbias_p99_pct",
         "qbias_p999_pct"]
# V5 = ORACLE (le modele causal), noncausal = CorrDiff (la baseline).
LABELS = {"V6_prime": "V6'", "V5_causal": "ORACLE", "noncausal_v4": "CorrDiff"}

_BL_LOCAL = RESULTS_DIR / "baselines_3way.json"
_cands = [_BL_LOCAL]
if IN_COLAB:
    _d = Path(DRIVE_ROOT) / "oracle_v6_prime"
    _cands = sorted(_d.glob("seed_*/v6_prime_3way_final_*.json")) + _cands
_bl_path = next((p for p in _cands if p.exists()), None)

baselines, refs_status, ref_protocol = {}, "ABSENTES", None
if _bl_path is not None:
    _bl = json.load(open(_bl_path))
    baselines = {k: v for k, v in _bl.get("three_way", {}).items()
                 if k in LABELS}
    refs_status = "RECOMPUTED_IN_PROTOCOL"
    if not baselines:
        print(f"!! {_bl_path} ne contient pas de bloc `three_way` exploitable "
              f"(clefs vues : {sorted(_bl.get('three_way', {}))}).")
    # Copie locale : le prochain run n'aura plus besoin de Drive.
    if baselines and _bl_path != _BL_LOCAL:
        json.dump({"three_way": baselines, "source": str(_bl_path)},
                  open(_BL_LOCAL, "w"), indent=2, default=float)
    print(f"references relues : {_bl_path}")
    _rp = Path(DRIVE_ROOT) / "oracle_v6_prime/recomputed_pergrid_references.json"
    if IN_COLAB and _rp.exists():
        ref_protocol = json.load(open(_rp)).get("protocol")

if not baselines:
    # Repli : les seuils pre-enregistres du depot. Le prereg V6' les marque
    # lui-meme STALE — ils ont ete calcules avec une Conv A BUGGEE (quantile
    # scalaire au lieu du broadcast per-pixel). On les affiche pour ne pas
    # laisser la table vide, jamais pour en tirer un verdict.
    _pr = json.load(open("path_c_plus/audit/V6_PRIME_seuils_preregistered.json"))
    _t = _pr["targets_to_beat"]
    baselines = {
        "V5_causal":    {"conv_A_F1p99": float(_t["co_primary_1_per_gridpoint"]["v5_causal_seed42"]),
                         "conv_B_F1p99": float(_t["co_primary_2_pooled"]["v5_causal_seed42"])},
        "noncausal_v4": {"conv_A_F1p99": float(_t["co_primary_1_per_gridpoint"]["noncausal_v4"]),
                         "conv_B_F1p99": float(_t["co_primary_2_pooled"]["noncausal_v4"])},
    }
    refs_status = "STALE_ANCIENNE_METRIQUE"
    print("!! Table 3-way introuvable. Repli sur les seuils pre-enregistres,")
    print("!! calcules avec la Conv A BUGGEE (prereg V6', REFS_STALE_WARNING).")
    print("!! Aucun verdict co-primaire n'est recevable dans cet etat : relancer")
    print("!! les Cells 13-15 du notebook V6' pour regenerer la table in-protocol.")

# --- parite de protocole ---------------------------------------------------
# Comparer sans reevaluer n'est licite que si les reglages coincident. Un K ou
# un nombre de pas different change les metriques a modele EGAL.
_mine = {"K": K_VERDICT, "num_steps": NUM_STEPS, "n_test": N_TEST}
_ecarts = []
if ref_protocol:
    for _k in ("K", "num_steps", "n_test"):
        if _k in ref_protocol and int(ref_protocol[_k]) != int(_mine[_k]):
            _ecarts.append(f"{_k}: reference={ref_protocol[_k]} vs V8={_mine[_k]}")
    print(f"protocole de reference : {ref_protocol}")
else:
    print("protocole de reference non retrouve (recomputed_pergrid_references.json) :")
    print("  parite NON verifiee ; les valeurs attendues sont K=32, 24 pas, cfg 0.0.")
comparable = (refs_status == "RECOMPUTED_IN_PROTOCOL") and not _ecarts
if _ecarts:
    print("!! ECART DE PROTOCOLE - la comparaison n'est pas apples-to-apples :")
    for _e in _ecarts:
        print(f"!!   {_e}")

# --- table -----------------------------------------------------------------
rows = {"V8": res}
rows.update({LABELS[k]: v for k, v in baselines.items()})
_lower = {"rmse", "mae", "rapsd_distance", "crps"}          # plus bas = mieux
_abs0 = {"bias_mm", "rx1day_bias", "qbias_p99_pct", "qbias_p999_pct"}  # proche de 0
print()
print("=" * 78)
print(f"COMPARAISON ({refs_status}, Conv A per-pixel, composition mm)")
print("=" * 78)
print(f"{'Metrique':22s} " + " ".join(f"{n:>12s}" for n in rows))
for met in TABLE:
    vals = {n: rows[n].get(met, float("nan")) for n in rows}
    _fin = [n for n in vals if vals[n] == vals[n]]
    if not _fin:
        continue
    if len(_fin) < 2:
        best = None          # une seule valeur : rien a comparer, pas de "best"
    elif met in _abs0:
        best = min(_fin, key=lambda n: abs(vals[n]))
    elif met == "ssr":                       # calibration : ~1 est le mieux
        best = min(_fin, key=lambda n: abs(vals[n] - 1.0))
    elif met in _lower:
        best = min(_fin, key=lambda n: vals[n])
    else:
        best = max(_fin, key=lambda n: vals[n])
    _cells = " ".join(f"{vals[n]:12.4f}" if vals[n] == vals[n] else f"{'-':>12s}"
                      for n in rows)
    print(f"{met:22s} {_cells}" + (f"   <- {best}" if best else ""))

# --- verdict ---------------------------------------------------------------
# Bande pre-enregistree (critique froide 2026-07-05) : le DAG gele est une
# feature OOD ASSUMEE, une regression in-distribution de ~2-8 % est PREVUE et
# acceptee. La cible V8 est la PARITE ID, pas un gain ID.
BANDE_ID_PCT = 8.0


def _delta(met, ref_key):
    r = baselines.get(ref_key, {}).get(met, float("nan"))
    v = res.get(met, float("nan"))
    if r != r or v != v or abs(r) < 1e-9:
        return float("nan"), float("nan")
    return v - r, 100.0 * (v - r) / abs(r)


verdict = {"refs_status": refs_status, "comparable": bool(comparable),
           "ecarts_protocole": _ecarts, "metrics_v8": res,
           "baselines": baselines, "protocole_v8": _mine,
           "bande_id_pct": BANDE_ID_PCT, "deltas": {}}
print()
print("=== ECARTS vs ORACLE (V5) et CorrDiff ===")
for met in ("conv_A_F1p99", "conv_B_F1p99", "rmse"):
    for ref_key, nom in (("V5_causal", "ORACLE"), ("noncausal_v4", "CorrDiff")):
        d, pct = _delta(met, ref_key)
        if pct != pct:
            continue
        verdict["deltas"][f"{met}_vs_{nom}"] = {"abs": d, "pct": pct}
        _st = ("PARITE" if abs(pct) <= BANDE_ID_PCT
               else ("GAIN" if (pct > 0) != (met == "rmse") else "HORS_BANDE"))
        print(f"  {met:14s} vs {nom:9s} : {d:+.4f} ({pct:+.1f} %)  {_st}")

verdict["lecture"] = (
    "ATTRIBUTION : plusieurs interrupteurs V8 sont actifs simultanement, ce run "
    "ne dit RIEN sur la contribution de chacun — il faut les runs P1 a un seul "
    "changement. "
    "CIBLE : parite in-distribution + gain OOD, pas un gain ID. Une regression "
    "ID de quelques pour cent est PREVUE et acceptee : le DAG gele est une "
    "feature OOD assumee (critique froide 2026-07-05). "
    "STATUT : ce verdict n'est PAS final. Il le devient sur EC-Earth3, puis UNE "
    "SEULE FOIS sur le holdout NorESM2-MM. "
    "INCERTITUDE : une seule graine, pas d'intervalle de confiance par blocs — "
    "un ecart de 1-2 points n'est pas interpretable.")
if not comparable:
    verdict["lecture"] = ("COMPARAISON NON VALIDE (" + refs_status + "). "
                          + verdict["lecture"])
json.dump(verdict, open(RESULTS_DIR / "v8_verdict.json", "w"),
          indent=2, default=float)
print()
print(f"=== ECRIT {RESULTS_DIR / 'v8_verdict.json'} ===")
print(verdict["lecture"])